In [9]:

from pathlib import Path
import random
import cv2
import numpy as np

# ---------------- CONFIGURATION ----------------

INPUT_DIR = Path("class_03_remote")
OUTPUT_DIR = Path("class_03_remote")

ORIGINAL_COUNT = 20
TOTAL_IMAGES = 50
SEED = 42

EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

random.seed(SEED)
rng = np.random.default_rng(SEED)


# ---------------- TRANSFORMATIONS ----------------

def rotate(image, angle):
    h, w = image.shape[:2]
    matrix = cv2.getRotationMatrix2D(
        (w / 2, h / 2), angle, 1.0
    )
    return cv2.warpAffine(
        image, matrix, (w, h),
        borderMode=cv2.BORDER_REFLECT_101
    )


def lighting(image):
    # Brightness, contrast and gamma variation
    alpha = random.uniform(0.75, 1.25)
    beta = random.randint(-25, 25)

    result = cv2.convertScaleAbs(
        image, alpha=alpha, beta=beta
    )

    gamma = random.uniform(0.8, 1.2)
    table = np.array([
        ((i / 255.0) ** (1.0 / gamma)) * 255
        for i in range(256)
    ]).clip(0, 255).astype(np.uint8)

    return cv2.LUT(result, table)


def shift(image):
    h, w = image.shape[:2]
    dx = random.randint(-int(w * 0.08), int(w * 0.08))
    dy = random.randint(-int(h * 0.08), int(h * 0.08))

    matrix = np.float32([
        [1, 0, dx],
        [0, 1, dy]
    ])

    return cv2.warpAffine(
        image, matrix, (w, h),
        borderMode=cv2.BORDER_REFLECT_101
    )


def zoom(image):
    h, w = image.shape[:2]
    scale = random.uniform(0.85, 1.15)

    matrix = cv2.getRotationMatrix2D(
        (w / 2, h / 2),
        random.uniform(-5, 5),
        scale
    )

    return cv2.warpAffine(
        image, matrix, (w, h),
        borderMode=cv2.BORDER_REFLECT_101
    )


def perspective(image):
    # Mild perspective distortion, not a true new viewpoint
    h, w = image.shape[:2]
    amount = min(h, w) * random.uniform(0.01, 0.04)

    src = np.float32([
        [0, 0], [w - 1, 0],
        [w - 1, h - 1], [0, h - 1]
    ])

    dst = src + np.float32([
        [random.uniform(-amount, amount),
         random.uniform(-amount, amount)],
        [random.uniform(-amount, amount),
         random.uniform(-amount, amount)],
        [random.uniform(-amount, amount),
         random.uniform(-amount, amount)],
        [random.uniform(-amount, amount),
         random.uniform(-amount, amount)]
    ])

    matrix = cv2.getPerspectiveTransform(src, dst)

    return cv2.warpPerspective(
        image, matrix, (w, h),
        borderMode=cv2.BORDER_REFLECT_101
    )


def color_variation(image):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV).astype(
        np.float32
    )

    hsv[:, :, 1] *= random.uniform(0.8, 1.2)
    hsv[:, :, 2] *= random.uniform(0.85, 1.15)

    hsv[:, :, 1:] = np.clip(hsv[:, :, 1:], 0, 255)

    return cv2.cvtColor(
        hsv.astype(np.uint8), cv2.COLOR_HSV2BGR
    )


def blur_or_noise(image):
    if random.choice([True, False]):
        kernel = random.choice([3, 5])
        return cv2.GaussianBlur(
            image, (kernel, kernel), 0
        )

    noise = rng.normal(0, 4, image.shape)
    return np.clip(
        image.astype(np.float32) + noise,
        0, 255
    ).astype(np.uint8)


# ---------------- CREATE DIFFERENT VARIATIONS ----------------

def augment(image, variation_number):
    # Each generated image receives a different random seed.
    local_seed = SEED + variation_number * 1009
    local_rng = random.Random(local_seed)

    result = image.copy()

    # Randomize transformation order and combinations.
    operations = [
        lambda img: rotate(
            img, local_rng.uniform(-18, 18)
        ),
        lambda img: lighting(img),
        lambda img: shift(img),
        lambda img: zoom(img),
        lambda img: perspective(img),
        lambda img: color_variation(img),
        lambda img: blur_or_noise(img)
    ]

    # Use 2–4 transformations for each generated image.
    count = local_rng.randint(2, 4)
    chosen = local_rng.sample(operations, count)

    # Keep parameters reproducible while varying images.
    for operation in chosen:
        result = operation(result)

    return result


# ---------------- MAIN PROGRAM ----------------

def main():
    if not INPUT_DIR.exists():
        raise FileNotFoundError(
            f"Folder not found: {INPUT_DIR.resolve()}"
        )

    # Read ONLY the 20 originals. Never read generated files.
    originals = []

    for i in range(1, ORIGINAL_COUNT + 1):
        expected_stem = f"remote_{i:03d}"

        matches = [
            p for p in INPUT_DIR.iterdir()
            if p.is_file()
            and p.stem == expected_stem
            and p.suffix.lower() in EXTENSIONS
        ]

        if len(matches) != 1:
            raise ValueError(
                f"Expected one file named {expected_stem} "
                f"with a supported extension; found {len(matches)}."
            )

        image = cv2.imread(str(matches[0]))

        if image is None:
            raise ValueError(
                f"Could not read {matches[0]}"
            )

        originals.append(image)

    # Save generated images only after originals are loaded.
    generated_count = TOTAL_IMAGES - ORIGINAL_COUNT

    for offset in range(generated_count):
        # Cycle through all 20 originals.
        source_index = offset % ORIGINAL_COUNT
        source = originals[source_index]

        output_number = ORIGINAL_COUNT + offset + 1

        generated = augment(
            source, variation_number=output_number
        )

        output_path = (
            OUTPUT_DIR / f"remote_{output_number:03d}.jpg"
        )

        if not cv2.imwrite(str(output_path), generated):
            raise IOError(f"Could not save {output_path}")

        print(
            f"Original remote_{source_index + 1:03d} "
            f"-> remote_{output_number:03d}"
        )

    print("\nDataset generation complete!")
    print(f"Original images: {ORIGINAL_COUNT}")
    print(f"Generated images: {generated_count}")
    print(f"Total dataset images: {TOTAL_IMAGES}")
    print(f"Folder: {OUTPUT_DIR.resolve()}")


if __name__ == "__main__":
    main()


Original remote_001 -> remote_021
Original remote_002 -> remote_022
Original remote_003 -> remote_023
Original remote_004 -> remote_024
Original remote_005 -> remote_025
Original remote_006 -> remote_026
Original remote_007 -> remote_027
Original remote_008 -> remote_028
Original remote_009 -> remote_029
Original remote_010 -> remote_030
Original remote_011 -> remote_031
Original remote_012 -> remote_032
Original remote_013 -> remote_033
Original remote_014 -> remote_034
Original remote_015 -> remote_035
Original remote_016 -> remote_036
Original remote_017 -> remote_037
Original remote_018 -> remote_038
Original remote_019 -> remote_039
Original remote_020 -> remote_040
Original remote_001 -> remote_041
Original remote_002 -> remote_042
Original remote_003 -> remote_043
Original remote_004 -> remote_044
Original remote_005 -> remote_045
Original remote_006 -> remote_046
Original remote_007 -> remote_047
Original remote_008 -> remote_048
Original remote_009 -> remote_049
Original remot